# This the Titanic dataset without pipelines

In [16]:
import numpy as np 
import pandas as pd 

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

In [17]:
df = pd.read_csv('Titanic-Dataset.csv')

In [18]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [19]:
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)

In [20]:
#Step 1: train-test-split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2,random_state=0)

In [21]:
X_train.head() # we have removed survived

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
140,3,female,NaN,0,2,15.2458,C
439,2,male,31.0,0,0,10.5000,S
817,2,male,31.0,1,1,37.0042,C
378,3,male,20.0,0,0,4.0125,C
491,3,male,21.0,0,0,7.2500,S


In [22]:
y_test.head() #this is only survived column

495    0
648    0
278    0
31     1
255    1
Name: Survived, dtype: int64

In [23]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [44]:
#Apply SimpleImputer to fill missing values
si_age = SimpleImputer() # will fill it with mean by default for numerical columns
si_embarked = SimpleImputer(strategy='most_frequent') # will fill it with most frequent value for categorical columns

X_train_age = si_age.fit_transform(X_train[['Age']])
X_train_embarked_imp = si_embarked.fit_transform(X_train[['Embarked']])

X_test_age = si_age.transform(X_test[['Age']])
X_test_embarked_imp = si_embarked.transform(X_test[['Embarked']])


In [ ]:
#one hot encoding Sex and Embarked columns
ohe_sex = OneHotEncoder(sparse_output=False,handle_unknown='ignore')
ohe_embarked = OneHotEncoder(sparse_output=False,handle_unknown='ignore')

X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
X_train_embarked_ohe = ohe_embarked.fit_transform(X_train_embarked_imp)

X_test_sex = ohe_sex.transform(X_test[['Sex']])
X_test_embarked_ohe = ohe_embarked.transform(X_test_embarked_imp)

In [46]:
X_train_rem = X_train.drop(columns=['Sex','Age','Embarked'])

In [47]:
X_test_rem = X_test.drop(columns=['Sex','Age','Embarked'])

In [50]:
X_train_transformed = np.concatenate([
	X_train_rem.to_numpy(),
	np.asarray(X_train_age),
	np.asarray(X_train_sex),
	np.asarray(X_train_embarked_ohe)
], axis=1)

X_test_transformed = np.concatenate([
	X_test_rem.to_numpy(),
	np.asarray(X_test_age),
	np.asarray(X_test_sex),
	np.asarray(X_test_embarked_ohe)
], axis=1)

X_train_transformed.shape

(712, 10)

In [51]:
clf = DecisionTreeClassifier()
clf.fit(X_train_transformed,y_train)

DecisionTreeClassifier()

DecisionTreeClassifier()

In [52]:
y_pred = clf.predict(X_test_transformed)

In [54]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.770949720670391

In [55]:
import pickle

In [58]:
pickle.dump(ohe_sex,open('ohe_sex.pkl','wb'))
pickle.dump(ohe_embarked,open('ohe_embarked.pkl','wb'))
pickle.dump(clf,open('clf.pkl','wb'))

In [60]:
ohe_sex = pickle.load(open('ohe_sex.pkl','rb'))
ohe_embarked = pickle.load(open('ohe_embarked.pkl','rb'))
clf = pickle.load(open('clf.pkl','rb'))

In [61]:
# Assume user input
# Pclass/gender/age/SibSp/Parch/Fare/Embarked
test_input = np.array([2, 'male', 31.0, 0, 0, 10.5, 'S'],dtype=object).reshape(1,7)

In [62]:
test_input

array([[2, 'male', 31.0, 0, 0, 10.5, 'S']], dtype=object)

In [63]:
test_input_sex = ohe_sex.transform(test_input[:,1].reshape(1,1))

c:\Users\Sanjog Bhalla\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [64]:
test_input_sex

array([[0., 1.]])

In [65]:
test_input_embarked = ohe_embarked.transform(test_input[:,-1].reshape(1,1))

In [66]:
test_input_age = test_input[:,2].reshape(1,1)

In [67]:
test_input_transformed = np.concatenate((test_input[:,[0,3,4,5]],test_input_age,test_input_sex,test_input_embarked),axis=1)

In [68]:
test_input_transformed.shape

(1, 10)

In [69]:
clf.predict(test_input_transformed)

array([0])